# 7장. RAG 검색증강생성 기본

이 장은 PDF 교재의 `6. RAG(검색증강생성)` 전체와 `LLM/llm.ipynb`, `LLM/llm2.ipynb`의 RAG 실습을 연결하는 기본 개념 장입니다.

## 이 장에서 다루는 내용

1. RAG란 무엇인가
2. 검색 단계와 생성 단계
3. RAG의 주요 특징
4. RAG 구성요소
5. RAG 파이프라인
6. RAG 활용 사례
7. RAG 핵심 기술: 자연어처리, 토크나이저, 임베딩
8. 문서 로딩
9. 문서 분할과 청크
10. 임베딩 변환
11. 벡터스토어 저장
12. 질의 임베딩과 검색
13. 프롬프트 구성
14. 답변 생성
15. RAG의 장점과 한계
16. 원본 노트북의 RAG 실습 위치

RAG는 이후 FAISS, Chroma, URL RAG, CSV RAG, DB 챗봇 실습을 이해하기 위한 핵심 개념입니다.


## 7.1 RAG란?

RAG는 `Retrieval-Augmented Generation`의 약자로, 한국어로는 `검색증강생성`이라고 합니다.

RAG는 LLM이 답변을 생성하기 전에 외부 데이터에서 관련 정보를 먼저 검색하고, 그 검색 결과를 근거로 답변을 생성하는 방식입니다.

기본 구조:

```text
사용자 질문
  -> 관련 문서 검색
  -> 검색된 문서를 프롬프트에 포함
  -> LLM이 근거 기반 답변 생성
```

일반 LLM은 학습 시점 이후의 정보나 회사 내부 문서를 모를 수 있습니다. RAG는 이런 한계를 외부 문서 검색으로 보완합니다.


## 7.2 검색 단계와 생성 단계

PDF 교재는 RAG의 작동 원리를 두 단계로 나눕니다.

### 검색 단계, Retrieval

질문을 기반으로 외부 데이터 소스에서 관련 문서를 검색합니다.

외부 데이터 소스 예시:

- 데이터베이스
- PDF 파일
- 텍스트 파일
- 웹 크롤링 데이터
- 벡터 데이터베이스
- 사내 매뉴얼
- FAQ 문서

### 생성 단계, Generation

검색된 정보를 LLM 프롬프트에 넣고 답변을 생성합니다.

LLM은 검색된 문서를 context로 보고, 사용자 질문에 맞는 답변을 작성합니다.

중요한 점은 답변 생성 자체는 LLM이 하지만, 답변의 근거는 외부 검색 결과에서 가져온다는 것입니다.


## 7.3 RAG의 주요 특징

PDF 교재의 RAG 주요 특징은 다음과 같습니다.

| 특징 | 설명 |
|---|---|
| 외부 데이터 활용 | 모델 학습 데이터 외의 최신 정보나 내부 자료를 사용할 수 있습니다. |
| 검색과 생성 결합 | 검색 시스템과 언어 모델을 결합해 단일 LLM보다 높은 정확도를 기대할 수 있습니다. |
| 최신성과 확장성 | 최신 데이터나 업데이트된 정보를 즉시 반영할 수 있습니다. |
| 모듈형 구조 | 검색 시스템과 생성 모델을 독립적으로 최적화할 수 있습니다. |

PDF 교재는 검색 시스템 예시로 `FAISS`, `Chroma` 같은 벡터 데이터베이스를 언급하고, 생성 모델 예시로 GPT, BERT, Gemma2, Llama3 같은 다양한 LLM 또는 sLLM을 언급합니다.


## 7.4 RAG가 필요한 이유

RAG가 필요한 대표 이유는 다음과 같습니다.

### 1. 최신 정보 반영

LLM은 학습된 시점 이후의 정보를 모를 수 있습니다. RAG는 최신 문서를 검색해 이 문제를 줄입니다.

### 2. 내부 문서 활용

회사 내부 매뉴얼, 규정, 보고서, 제품 문서는 일반 LLM이 알 수 없습니다. RAG를 사용하면 내부 문서 기반 Q/A를 만들 수 있습니다.

### 3. 환각 감소

LLM은 모르는 내용을 그럴듯하게 만들어낼 수 있습니다. RAG는 검색된 근거를 제공해 사실 기반 답변을 유도합니다.

### 4. 모델 재학습 없이 지식 확장

파인튜닝은 모델 가중치를 다시 학습해야 하지만, RAG는 문서 저장소만 업데이트하면 됩니다.

### 5. 근거 제시 가능

검색된 문서 출처를 함께 보여주면 답변의 신뢰성을 높일 수 있습니다.


## 7.5 RAG 구성요소

PDF 6.2의 RAG 구성요소를 텍스트로 정리하면 다음과 같습니다.

| 구성요소 | 역할 |
|---|---|
| 원본 문서 | PDF, TXT, CSV, 웹페이지, DB 등 검색 대상 데이터입니다. |
| Document Loader | 원본 자료를 LangChain 문서 객체로 읽습니다. |
| Text Splitter | 긴 문서를 작은 청크로 나눕니다. |
| Embedding Model | 문서 청크와 질문을 벡터로 변환합니다. |
| Vector Store | 문서 벡터와 원문 청크를 저장하고 검색합니다. |
| Retriever | 질문과 관련 있는 문서 청크를 찾습니다. |
| Prompt Template | 검색된 문서와 질문을 LLM 입력 형태로 구성합니다. |
| LLM | 검색된 context를 바탕으로 답변을 생성합니다. |
| Output Parser | LLM 결과를 문자열 등 원하는 형태로 정리합니다. |

LangChain RAG 코드는 이 구성요소들을 연결하는 형태로 작성됩니다.


## 7.6 RAG 파이프라인 전체 흐름

PDF 6.3과 6.7의 RAG 파이프라인은 다음 8단계로 정리할 수 있습니다.

```text
1. 문서 로딩
2. 문서 분할
3. 임베딩 변환
4. 벡터스토어 저장
5. 사용자 질의 입력
6. 질의 임베딩 및 검색
7. 프롬프트 구성
8. 답변 생성
```

이 8단계는 `llm.ipynb`의 FAISS/Chroma RAG 예제와 `llm2.ipynb`의 CSV/URL RAG 예제에서 반복됩니다.


## 7.7 1단계: 문서 로딩

문서 로딩은 PDF, 텍스트 파일, 웹페이지, CSV 같은 원본 자료를 시스템으로 읽어오는 단계입니다.

LangChain에서 자주 사용하는 Loader 예시:

| Loader | 사용 대상 |
|---|---|
| `TextLoader` | `.txt` 파일 |
| `WebBaseLoader` | 웹페이지 URL |
| CSV 관련 로더 또는 Pandas | `.csv` 파일 |
| PDF Loader | PDF 문서 |

`llm.ipynb`의 FAISS RAG 예제는 다음 코드를 사용합니다.

```python
loader = TextLoader("./dataset/history.txt", encoding='UTF8')
documents = loader.load()
```

여기서 `documents`는 문서 내용과 metadata를 포함하는 LangChain Document 객체 목록입니다.


## 7.8 2단계: 문서 분할

긴 문서를 그대로 임베딩하거나 프롬프트에 넣으면 문제가 생깁니다.

- 너무 긴 문서는 임베딩 품질이 떨어질 수 있습니다.
- LLM의 컨텍스트 길이 제한에 걸릴 수 있습니다.
- 질문과 관련 있는 작은 부분만 찾기 어렵습니다.

그래서 문서를 작은 청크로 나눕니다.

PDF 교재는 청크 간 의미적 연결성을 유지하기 위해 약간의 overlap을 둔다고 설명합니다.

주요 개념:

| 개념 | 설명 |
|---|---|
| chunk | 문서를 나눈 작은 텍스트 조각 |
| chunk_size | 청크 하나의 최대 길이 |
| chunk_overlap | 이전 청크와 다음 청크가 겹치는 길이 |

원본 노트북의 Chroma 예제:

```python
RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len
)
```


## 7.9 3단계: 임베딩 변환

임베딩은 텍스트의 의미를 숫자 벡터로 바꾸는 과정입니다.

RAG에서는 두 종류의 텍스트를 임베딩합니다.

1. 문서 청크
2. 사용자 질문

문서 청크와 질문을 같은 임베딩 모델로 벡터화해야 같은 벡터 공간에서 유사도를 비교할 수 있습니다.

원본 노트북에서 사용한 임베딩 모델 예시:

| 임베딩 모델 | 사용 위치 |
|---|---|
| `sentence-transformers/all-MiniLM-L6-v2` | FAISS RAG, Chroma RAG, CSV RAG |
| `jhgan/ko-sroberta-multitask` | 간단한 한국어 LangChain 검색 예제 |
| `OllamaEmbeddings(model="gemma2")` | URL 기반 RAG |
| `BAAI/bge-m3` | Chroma 서버 예제의 추천 모델 주석 |

임베딩 모델 선택은 검색 품질에 큰 영향을 줍니다. 한국어 문서라면 한국어 또는 다국어에 강한 임베딩 모델을 고려해야 합니다.


## 7.10 4단계: 벡터스토어 저장

벡터스토어는 임베딩 벡터와 원본 텍스트 청크를 저장하고, 질문 벡터와 비슷한 문서를 빠르게 찾아주는 저장소입니다.

원본 노트북에서 사용한 벡터스토어:

- FAISS
- Chroma

FAISS 예시:

```python
vectorstore = FAISS.from_documents(documents, embeddings)
```

Chroma 예시:

```python
vectordb = Chroma.from_documents(
    documents=texts,
    embedding=embeddings,
    persist_directory=PERSIST_DIRECTORY
)
vectordb.persist()
```

FAISS는 빠른 벡터 검색 라이브러리이고, Chroma는 개발 환경에서 쓰기 쉬운 영속성 벡터 DB입니다.


## 7.11 5단계: 사용자 질의 입력

RAG 시스템은 사용자의 질문을 입력으로 받습니다.

원본 노트북의 질문 예시:

```python
query = "고조선은 언제 설립되었는지 알려줘."
```

CSV RAG 예시 질문:

```text
한국폴리텍대학 스마트금융과 입학 전까지 어떤걸 공부하면 될까요?
스마트금융과에 대해 설명해주세요
한국폴리텍대한 추천할만한 학과 하나를 소개해주세요.
```

질문은 그대로 LLM에 전달되는 동시에, Retriever가 관련 문서를 찾는 데 사용됩니다.


## 7.12 6단계: 질의 임베딩 및 검색

사용자 질문도 문서와 같은 방식으로 임베딩 벡터로 변환됩니다.

그 다음 벡터스토어에 저장된 문서 청크 벡터들과 유사도를 비교합니다.

PDF 교재는 주로 코사인 유사도를 언급합니다.

검색 결과는 보통 상위 K개 문서 청크입니다.

```python
retriever = vectorstore.as_retriever()
docs = retriever.invoke(query)
```

또는 검색 개수를 지정할 수 있습니다.

```python
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})
```

`k`가 너무 작으면 필요한 문서가 빠질 수 있고, 너무 크면 LLM 프롬프트가 길어지거나 불필요한 문맥이 섞일 수 있습니다.


## 7.13 7단계: 프롬프트 구성

검색된 문서 청크를 LLM에게 그냥 따로 보내는 것이 아니라, 질문과 함께 프롬프트로 구성해야 합니다.

원본 노트북의 대표 프롬프트:

```text
당신은 질문에 답변하는 AI 어시스턴트입니다.
제공된 context만을 바탕으로 질문에 답변하세요. 모르면 모른다고 답하세요.

[Context]
{context}

[Question]
{question}
```

이 프롬프트는 두 가지 중요한 규칙을 포함합니다.

1. 제공된 context만 기반으로 답변합니다.
2. 모르면 모른다고 답합니다.

이 규칙은 RAG에서 환각을 줄이는 데 중요합니다.


## 7.14 8단계: 답변 생성

마지막 단계는 LLM이 프롬프트를 읽고 답변을 생성하는 것입니다.

원본 노트북은 주로 Ollama Gemma2 모델을 사용합니다.

```python
llm = Ollama(model="gemma2", base_url="http://localhost:11434")
```

LCEL 체인 구성 예시:

```python
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
```

실행:

```python
response = rag_chain.invoke(query)
```

이 구조는 이후 FAISS, Chroma, URL RAG 장에서 계속 반복됩니다.


## 7.15 RAG 활용 사례

PDF 6.4의 RAG 활용 사례는 다음 세 가지입니다.

### 1. 문서 기반 질의응답

PDF, 워드 문서, 데이터베이스 등에서 정보를 검색해 정확한 답변을 제공합니다.

예시:

- 회사 내부 보고서를 바탕으로 질문응답
- 사내 규정 검색 챗봇
- 교육 자료 기반 Q/A

### 2. 실시간 정보 제공

최신 뉴스나 제품 정보를 검색해 질문에 응답합니다.

예시:

- 오늘 날씨 질문
- 최신 제품 스펙 질문
- 웹페이지 URL 기반 요약

### 3. 헬프데스크/챗봇

FAQ나 매뉴얼 데이터를 검색한 후 고객에게 답변합니다.

예시:

- 제품 사용 방법 안내
- 고객센터 자동 응답
- 장애 해결 가이드 챗봇


## 7.16 RAG 핵심 기술: 자연어처리 과정

PDF 6.5는 RAG의 핵심 기술로 자연어처리 과정을 다시 설명합니다.

RAG에서도 1장에서 다룬 NLP 과정이 그대로 중요합니다.

```text
텍스트 -> 토큰화 -> 인코딩 -> 임베딩 -> 벡터 검색
```

문서와 질문은 모두 텍스트입니다. 텍스트를 벡터로 바꾸지 않으면 의미 기반 검색을 할 수 없습니다.

RAG에서 자연어처리 과정은 다음 위치에 등장합니다.

- 문서 청크를 임베딩할 때
- 사용자 질문을 임베딩할 때
- 검색된 context를 프롬프트에 넣을 때
- LLM이 답변을 생성할 때


## 7.17 RAG 핵심 기술: 토크나이저

토크나이저는 원문 텍스트를 모델이 처리할 수 있는 작은 단위로 나눕니다.

RAG에서 토크나이저가 중요한 이유:

- 임베딩 모델은 내부적으로 텍스트를 토큰화합니다.
- LLM은 context와 question을 토큰 단위로 처리합니다.
- 모델마다 최대 토큰 길이가 있습니다.
- 문서 청크가 너무 길면 토큰 제한을 초과할 수 있습니다.

따라서 RAG에서는 청크 크기를 문자 수만이 아니라 토큰 수 관점에서도 고려해야 합니다.


## 7.18 RAG 핵심 기술: 임베딩

임베딩은 RAG의 검색 품질을 결정하는 핵심 요소입니다.

좋은 임베딩 모델은 의미가 비슷한 문장을 벡터 공간에서 가깝게 배치합니다.

예시:

```text
고조선은 언제 세워졌나요?
고조선의 건국 시기는 언제인가요?
```

두 문장은 표현은 다르지만 의미가 비슷합니다. 좋은 임베딩 모델은 두 질문의 벡터를 가깝게 만듭니다.

임베딩 모델 선택 기준:

- 한국어 문서인지 영어 문서인지
- 문장 단위 검색인지 문서 단위 검색인지
- 속도와 정확도 균형
- 로컬 실행 가능 여부
- 다국어 지원 여부


## 7.19 원본 노트북의 RAG 실습 정리

두 원본 노트북에는 여러 RAG 실습이 들어 있습니다.

| 파일 | 실습 | 주요 기술 |
|---|---|---|
| `llm.ipynb` | 간단한 LangChain + FAISS 검색 예제 | FAISS, HuggingFaceEmbeddings, Ollama |
| `llm.ipynb` | Ollama + Gemma2 + FAISS RAG | TextLoader, FAISS, LCEL |
| `llm.ipynb` | Ollama + Gemma2 + Chroma RAG | Chroma, persist_directory |
| `llm.ipynb` | Chroma 서버 접근 RAG | Chroma HttpClient, 원격 벡터 DB |
| `llm2.ipynb` | CSV 기반 Gradio RAG 챗봇 | Pandas, CharacterTextSplitter, FAISS |
| `llm2.ipynb` | URL 기반 RAG 챗봇 | WebBaseLoader, RecursiveCharacterTextSplitter, OllamaEmbeddings |

다음 장부터는 이 실습들을 구체적으로 하나씩 구현합니다.


## 7.20 RAG의 장점

RAG의 장점은 다음과 같습니다.

### 1. 최신 정보 반영

문서 저장소만 업데이트하면 최신 정보를 답변에 반영할 수 있습니다.

### 2. 모델 재학습 불필요

새 지식을 모델 파라미터에 넣기 위해 Fine-tuning하지 않아도 됩니다.

### 3. 근거 기반 답변

검색된 문서 내용을 context로 사용하므로 답변 근거를 제시할 수 있습니다.

### 4. 도메인 지식 활용

사내 문서, 제품 매뉴얼, 규정집처럼 모델이 모르는 자료를 활용할 수 있습니다.

### 5. 모듈별 개선 가능

임베딩 모델, 벡터스토어, Retriever, Prompt, LLM을 각각 교체하거나 최적화할 수 있습니다.


## 7.21 RAG의 한계

RAG는 강력하지만 완벽하지 않습니다.

### 1. 검색 실패 문제

관련 문서를 검색하지 못하면 LLM은 좋은 답을 만들기 어렵습니다.

### 2. 잘못된 문서 검색

관련 없는 문서가 검색되면 답변도 틀릴 수 있습니다.

### 3. 문서 품질 문제

원본 문서가 부정확하거나 오래된 정보라면 답변도 부정확해집니다.

### 4. 청크 분할 문제

청크가 너무 작으면 문맥이 끊기고, 너무 크면 검색 정확도가 낮아질 수 있습니다.

### 5. 프롬프트 설계 문제

LLM에게 context만 기반으로 답하라고 명확히 지시하지 않으면 모델 자체 지식이나 추측이 섞일 수 있습니다.

### 6. 비용과 속도

검색, 임베딩, LLM 호출이 모두 들어가므로 단순 LLM 호출보다 복잡하고 느릴 수 있습니다.


## 7.22 좋은 RAG를 만들기 위한 체크리스트

- [ ] 원본 문서가 정확하고 최신인가?
- [ ] 문서 로딩이 제대로 되었는가?
- [ ] 청크 크기와 overlap이 적절한가?
- [ ] 문서 언어에 맞는 임베딩 모델을 사용했는가?
- [ ] 벡터스토어 검색 결과가 질문과 관련 있는가?
- [ ] 검색 개수 `k`가 적절한가?
- [ ] 프롬프트에 `context만 기반으로 답하라`는 지시가 있는가?
- [ ] 모르면 모른다고 답하도록 했는가?
- [ ] 답변에 출처를 표시할 수 있는가?
- [ ] 검색 실패 시 사용자에게 적절히 안내하는가?
- [ ] Ollama 또는 LLM 서버가 안정적으로 실행되는가?
- [ ] Gradio UI에서 오류가 사용자에게 이해되게 표시되는가?


## 7.23 Fine-tuning, Distillation, RAG의 차이 미리보기

PDF 9.2는 Fine-tuning, Distillation, RAG를 비교합니다. 자세한 내용은 19장에서 다루지만, RAG 장에서 핵심 차이를 미리 보면 다음과 같습니다.

| 구분 | Fine-tuning | Distillation | RAG |
|---|---|---|---|
| 지식의 위치 | 모델 내부 파라미터 | 작은 모델 내부 | 모델 외부 문서/벡터 DB |
| 모델 업데이트 | 필요 | 필요 | 불필요 |
| 주요 목적 | 특정 도메인 전문화 | 모델 경량화 | 최신 정보 반영, 근거 기반 답변 |
| 장점 | 일관된 스타일과 전문성 | 비용과 속도 개선 | 문서만 바꿔 지식 업데이트 가능 |
| 단점 | 데이터 준비와 재학습 필요 | teacher보다 성능 저하 가능 | 검색 품질에 크게 의존 |

이 교재의 실습 대부분은 RAG와 Prompt/Chain 중심입니다. 이유는 모델을 재학습하지 않고도 로컬 문서와 외부 데이터를 연결해 실용적인 챗봇을 만들 수 있기 때문입니다.


## 7.24 이 장의 핵심 정리

이 장에서 배운 내용을 정리하면 다음과 같습니다.

- RAG는 검색증강생성입니다.
- RAG는 외부 문서를 검색하고, 검색된 정보를 바탕으로 LLM이 답변을 생성합니다.
- RAG는 검색 단계와 생성 단계로 나뉩니다.
- RAG는 최신 정보, 내부 문서, 근거 기반 답변에 유리합니다.
- RAG의 8단계는 문서 로딩, 문서 분할, 임베딩, 벡터스토어 저장, 질의 입력, 검색, 프롬프트 구성, 답변 생성입니다.
- 문서 청크와 사용자 질문은 같은 임베딩 공간에서 비교됩니다.
- Retriever는 질문과 관련 있는 문서 청크를 찾습니다.
- Prompt Template은 검색된 context와 질문을 LLM 입력으로 구성합니다.
- LLM은 검색된 context를 바탕으로 최종 답변을 생성합니다.
- RAG의 품질은 문서 품질, 청크 전략, 임베딩 모델, 검색 품질, 프롬프트 설계에 크게 좌우됩니다.

다음 장에서는 FAISS 기반 RAG 실습을 실제 코드 중심으로 구현합니다.
